# LSTM with SHAP & Masking - subsurface_Layer01

In [1]:
# imports and Configuration
import os
import sys
import psutil
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import pickle, json, time, warnings
warnings.filterwarnings('ignore')
from google.colab import drive

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix,
    accuracy_score, average_precision_score,
    classification_report
)
import math
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import xarray as xr

from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss

import shap  # for feature importance

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# configuration
CONFIG_NAME = 'subsurface_layer_0-50m_layer'
FEATURES = ['sst', 'wind_speed','ssr_flux', 'str_flux', 'latent_flux', 'sensible_flux', 't2m', 'msl','temp_0_50m', 'temp_50_100m', 'temp_100_150m']
TARGET = 'mhw_layer1'
LEAD_TIMES = [3, 5, 7, 9, 12, 14]

# LSTM parameters
# LSTM_PARAMS = {
#     'window_size': 14,
#     'hidden_units': [128, 64],
#     'dropout': 0.3,
#     'batch_size': 2048,
#     'epochs': 30,
#     'learning_rate': 1e-3,
#     'mask_value': -10
# }

LSTM_PARAMS = {
    'window_size':   15,
    'batch_size':    2048,
    'hidden_units':  [64,32],   # ← update
    'dropout':       0.5,          # ← update
    'learning_rate': 1e-4,
    'epochs':        100,
    'mask_value': -10

}

print('='*80)
print(f'LSTM with SHAP & Masking: {CONFIG_NAME}')
print('='*80)

LSTM with SHAP & Masking: subsurface_layer_0-50m_layer


In [2]:
# Verify GPU is working:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs: {gpus}")


GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# output directory
OUTPUT_DIR = Path('/content/drive/MyDrive/MSc-research-code/trained_models/lstm/lstm_subsurface_layer_0-50m_layer')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)


In [ ]:
# #Enable GPU
# print('Checking GPU...')
# import tensorflow as tf

# gpus = tf.config.list_physical_devices('GPU')
# if not gpus:
#     print('NO GPU! Please enable it:')
#     # print('   Runtime → Change runtime type → T4 GPU → Save → Restart')
#     raise RuntimeError('GPU not enabled!')

# print(f'GPU enabled: {gpus[0].name}')

# # Enable memory growth
# for gpu in gpus:
#     tf.config.experimental.set_memory_growth(gpu, True)

In [5]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU ENABLED: {len(gpus)} device(s)")
    for gpu in gpus:
        print(f"  {gpu}")

    # Mixed precision for 2× speedup
    mixed_precision.set_global_policy('mixed_float16')
    print("Mixed precision enabled (2× faster training)")
else:
    print(" NO GPU - Training will be VERY slow!")
    print("   Consider using subset or getting GPU access")

print("="*80)

GPU ENABLED: 1 device(s)
  PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
Mixed precision enabled (2× faster training)


In [6]:

def xr_to_df(ds, features, targets):
    """Convert xarray Dataset to pandas DataFrame."""
    print('Converting xarray → pandas...')
    stacked = ds[features + targets].stack(sample=('time', 'latitude', 'longitude'))
    df = stacked.to_dataframe().reset_index()

    initial_len = len(df)
    df = df.dropna(subset=['sst'])
    print(f'  Removed {initial_len - len(df):,} land points')

    df = df.sort_values(['latitude', 'longitude', 'time']).reset_index(drop=True)
    df['location'] = df['latitude'].astype(str) + '_' + df['longitude'].astype(str)

    for col in features:
        if col in df.columns:
            df[col] = df[col].astype(np.float32)
    for col in targets:
        if col in df.columns:
            df[col] = df[col].astype(np.int8)

    print(f'  Samples: {len(df):,}')
    return df


In [7]:
NETCDF_FILE = '/content/drive/MyDrive/MSc-research-code/data/ocean_atmos_DAILY_WITH_LABELS_1993_2025.nc'

print('Loading NetCDF...')
ds = xr.open_dataset(NETCDF_FILE)
df = xr_to_df(ds, FEATURES, [TARGET])
ds.close()

print(f'Data loaded: {df.shape}')
print('='*80)

Loading NetCDF...
Converting xarray → pandas...
  Removed 3,061,462 land points
  Samples: 11,703,463
Data loaded: (11703463, 18)


In [8]:
TRAIN_END_DATE = '2019-12-31'
VAL_START_DATE = '2020-01-15'
VAL_END_DATE = '2022-12-31'
TEST_START_DATE = '2023-01-15'

df['time'] = pd.to_datetime(df['time'])
train_df = df[df['time'] <= TRAIN_END_DATE].copy()
val_df = df[(df['time'] >= VAL_START_DATE) & (df['time'] <= VAL_END_DATE)].copy()
test_df = df[df['time'] >= TEST_START_DATE].copy()

print(f'Train: {len(train_df):,} samples')
print(f'Validation: {len(val_df):,} samples')
print(f'Test:  {len(test_df):,} samples')

del df
import gc; gc.collect()
print('='*80)

Train: 9,575,031 samples
Validation: 1,050,622 samples
Test:  1,050,622 samples


In [9]:

def create_multi_output_targets(df, target_col, lead_times):
    """
    Create multi-output targets for a specific MHW variable.

    Citation: Ben Taieb & Atiya (2016) DOI: 10.1109/TNNLS.2015.2411629
    """
    df = df.copy().sort_values(['location', 'time']).reset_index(drop=True)

    target_cols = []
    for lead in lead_times:
        col_name = f'{target_col}_lead{lead}'
        # Shift target FORWARD by lead days (negative shift)
        df[col_name] = df.groupby('location', sort=False)[target_col].shift(-lead)
        target_cols.append(col_name)

    # Remove rows with NaN in any target
    initial_len = len(df)
    df = df.dropna(subset=target_cols)
    dropped = initial_len - len(df)

    print(f'  Target: {target_col:15s} | Created: {len(target_cols)} | '
          f'Samples: {len(df):,} (dropped {dropped:,})')

    return df, target_cols

# Create targets for EACH configuration
print('Creating Multi-Output Targets')
print('='*80)

train_targets = {}  # {config_name: (dataframe, target_cols)}
val_targets = {}
test_targets = {}

# for config_name, config_dict in DEPTH_CONFIGS.items():
#     target = config_dict['target']
#     print(f'\n{config_name}:')

# Train
train_df_config, train_cols = create_multi_output_targets(
  train_df, TARGET, LEAD_TIMES
)
train_targets = (train_df_config, train_cols)

# Val
val_df_config, val_cols = create_multi_output_targets(
  val_df, TARGET, LEAD_TIMES
)
val_targets = (val_df_config, val_cols)

# Test
test_df_config, test_cols = create_multi_output_targets(
  test_df, TARGET, LEAD_TIMES
)
test_targets = (test_df_config, test_cols)

print('\n' + '='*80)
print('All multi-output targets created')
print('='*80)

Creating Multi-Output Targets
  Target: mhw_layer1      | Created: 6 | Samples: 9,561,437 (dropped 13,594)
  Target: mhw_layer1      | Created: 6 | Samples: 1,037,028 (dropped 13,594)
  Target: mhw_layer1      | Created: 6 | Samples: 1,037,028 (dropped 13,594)

All multi-output targets created


In [10]:
class MaskedLSTMDataset(tf.keras.utils.Sequence):
    """
    Memory-efficient LSTM dataset.

    Key points:
    - Converts DataFrame to NumPy ONCE
    - Stores loc_indices per sequence (gap safe)
    - RobustScaler handles NaN natively
      NaN in → NaN out after transform
    - Applies mask_value=-10.0 AFTER scaling
    """

    def __init__(self, df, features, target_col,
                 window_size, batch_size,
                 scaler=None, mask_value=-10.0,
                 shuffle=True):

        df_sorted = df.sort_values(
            ['location', 'time']
        ).reset_index(drop=True)

        print('  Converting DataFrame to NumPy...')
        self.X_data    = df_sorted[features].values.astype(np.float32)
        self.y_data    = df_sorted[target_col].values.astype(np.float32)
        self.locations = df_sorted['location'].values

        self.features    = features
        self.n_features  = len(features)
        self.target_col  = target_col
        self.window_size = window_size
        self.batch_size  = batch_size
        self.scaler      = scaler
        self.mask_value  = mask_value   # -10.0
        self.shuffle     = shuffle

        # Build sequence indices
        self.sequence_indices = []
        print('  Building sequence indices...')
        unique_locs    = np.unique(self.locations)
        location_count = 0

        for location in unique_locs:
            location_count += 1
            if location_count % 1000 == 0:
                print(f'    {location_count:,} locations | '
                      f'{len(self.sequence_indices):,} sequences')

            loc_mask    = self.locations == location
            loc_indices = np.where(loc_mask)[0]
            n           = len(loc_indices)

            if n <= window_size:
                continue

            for i in range(n - window_size):
                target_idx = loc_indices[i + window_size - 1]

                if np.isnan(self.y_data[target_idx]):
                    continue

                # Store actual indices — gap safe
                self.sequence_indices.append((
                    loc_indices[i:i + window_size].copy(),
                    target_idx
                ))

        self.n_sequences = len(self.sequence_indices)

        X_mb   = self.X_data.nbytes / 1024**2
        y_mb   = self.y_data.nbytes / 1024**2
        seq_mb = (self.n_sequences * window_size * 8) / 1024**2

        print(f'  Dataset ready: {self.n_sequences:,} sequences')
        print(f'    X array:  {X_mb:.1f} MB')
        print(f'    y array:  {y_mb:.1f} MB')
        print(f'    Indices:  {seq_mb:.1f} MB')
        print(f'    Batches:  {len(self):,}')

        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(self.n_sequences / self.batch_size))

    def __getitem__(self, idx):
        batch_start       = idx * self.batch_size
        batch_end         = min((idx + 1) * self.batch_size,
                                self.n_sequences)
        batch_meta        = self.sequence_indices[batch_start:batch_end]
        batch_size_actual = len(batch_meta)

        X_batch = np.empty(
            (batch_size_actual, self.window_size, self.n_features),
            dtype=np.float32
        )
        y_batch = np.empty(batch_size_actual, dtype=np.float32)

        # Gap-safe indexing
        for i, (loc_indices, target_idx) in enumerate(batch_meta):
            X_batch[i] = self.X_data[loc_indices]
            y_batch[i] = self.y_data[target_idx]

        # Step 1 — record NaN positions BEFORE anything
        nan_mask = np.isnan(X_batch)

        if self.scaler is not None:
            original_shape = X_batch.shape
            X_flat         = X_batch.reshape(-1, self.n_features)

            # Step 2 — transform directly
            # RobustScaler handles NaN natively
            # NaN in -> NaN out (no fill needed)
            with warnings.catch_warnings():
                warnings.filterwarnings('ignore')
                X_flat_scaled = self.scaler.transform(X_flat)

            X_batch = X_flat_scaled.reshape(original_shape)

        #  apply -10.0 to NaN positions AFTER scaling
        # nan_mask still knows original NaN positions
        # -10.0 is outside RobustScaler range [-3, +3]
        X_batch[nan_mask] = self.mask_value

        y_batch = y_batch.reshape(-1, 1)
        return X_batch, y_batch

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.sequence_indices)


print('='*80)
print('MaskedLSTMDataset defined')
print('='*80)

MaskedLSTMDataset defined


In [11]:
def build_masked_lstm(input_shape,
                      hidden_units=[64,32],
                      dropout=0.5,
                      mask_value=-10.0):
    """
    cuDNN-compatible LSTM.
    No Masking layer — cuDNN does not support it.
    -10.0 sentinel handled as learned pattern.
    """
    inputs = keras.Input(shape=input_shape)

    # First LSTM — cuDNN requirements
    x = layers.LSTM(
        hidden_units[0],
        return_sequences    = True,
        activation          = 'tanh',       # required for cuDNN
        recurrent_activation= 'sigmoid',    # required for cuDNN
        recurrent_dropout   = 0,            # MUST be 0 for GPU
        unroll              = False
    )(inputs)
    x = layers.Dropout(dropout)(x)

    # Second LSTM
    x = layers.LSTM(
        hidden_units[1],
        return_sequences    = False,
        activation          = 'tanh',
        recurrent_activation= 'sigmoid',
        recurrent_dropout   = 0,
        unroll              = False
    )(x)
    x = layers.Dropout(dropout)(x)

    x = layers.Dense(16, activation='relu')(x)
    x = layers.BatchNormalization()(x)      # training stability

    # Binary output — float32 for mixed precision
    outputs = layers.Dense(
        1, activation='sigmoid', dtype='float32'
    )(x)

    return keras.Model(inputs=inputs, outputs=outputs)


print('build_masked_lstm defined')


build_masked_lstm defined


In [12]:
def focal_loss(gamma=2.0, alpha=0.9):
    """
    Focal loss for imbalanced binary classification.
    alpha computed as 1 - mhw_rate per lead time.
    """
    def loss_fn(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred  = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        cross_entropy = (
            -y_true * tf.math.log(y_pred)
            -(1 - y_true) * tf.math.log(1 - y_pred)
        )

        p_t          = tf.where(tf.equal(y_true, 1),
                                y_pred, 1 - y_pred)
        focal_weight = tf.pow(1.0 - p_t, gamma)
        alpha_weight = tf.where(tf.equal(y_true, 1),
                                alpha, 1 - alpha)

        return tf.reduce_mean(
            alpha_weight * focal_weight * cross_entropy
        )
    return loss_fn


print('focal_loss defined')

focal_loss defined


In [13]:
def sedi(tp, fp, fn, tn):
    """
    Symmetric Extremal Dependence Index.
    Community standard metric for rare MHW events.
    SEDI = 0 → no skill above random
    SEDI = 1 → perfect forecast
    """
    H = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    F = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    eps = 1e-7
    H = np.clip(H, eps, 1 - eps)
    F = np.clip(F, eps, 1 - eps)

    return float(
        (np.log(F) - np.log(H) -
         np.log(1-F) + np.log(1-H))
        /
        (np.log(F) + np.log(H) +
         np.log(1-F) + np.log(1-H))
    )


def bss(y_true, y_prob, climatology_rate=None):
    """
    Brier Skill Score vs climatology.
    BSS > 0 → better than climatology
    BSS = 0 → same as climatology
    BSS < 0 → worse than climatology
    BSS = nan → undefined (all same class)
    """
    bs_model = np.mean((y_prob - y_true) ** 2)

    if climatology_rate is None:
        climatology_rate = y_true.mean()

    bs_clim = np.mean((climatology_rate - y_true) ** 2)

    if bs_clim == 0:
        return np.nan    # undefined — all same class

    return float(1 - bs_model / bs_clim)

In [14]:
# ==============================================================================
# compute_metrics
# ==============================================================================

def compute_metrics(y_test, y_pred, y_prob, threshold,
                    strategy, model_name, config,
                    target, lead):
    """
    Identical to LightGBM compute_metrics.
    Enables direct CSV comparison between models.
    """
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    return {
        'model':              model_name,
        'config':             config,
        'target':             target,
        'lead_time':          lead,
        'threshold_strategy': strategy,
        'best_threshold':     float(threshold),
        'auc':                roc_auc_score(y_test, y_prob),
        'avg_precision':      average_precision_score(y_test, y_prob),
        'f1':                 f1_score(y_test, y_pred, zero_division=0),
        'precision':          precision_score(y_test, y_pred, zero_division=0),
        'recall':             recall_score(y_test, y_pred, zero_division=0),
        'f1_0':               f1_score(y_test, y_pred,
                                       pos_label=0, zero_division=0),
        'precision_0':        precision_score(y_test, y_pred,
                                              pos_label=0, zero_division=0),
        'recall_0':           recall_score(y_test, y_pred,
                                           pos_label=0, zero_division=0),
        'accuracy':           accuracy_score(y_test, y_pred),
        'macro_f1':           f1_score(y_test, y_pred,
                                       average='macro', zero_division=0),
        'tn':  int(tn), 'fp': int(fp),
        'fn':  int(fn), 'tp': int(tp),
        'tpr': float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0,
        'tnr': float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0,
        'fpr': float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0,
        'csi': float(tp / (tp + fp + fn)) if (tp+fp+fn) > 0 else 0.0,
        'sedi': sedi(tp, fp, fn, tn),
        'bss':  bss(y_test, y_prob),
        'mhw_rate_test': float(y_test.mean())
    }


print('compute_metrics defined')

compute_metrics defined


In [ ]:
SKIP_LEADS = []


def get_memory_usage():
    process = psutil.Process()
    mem     = process.memory_info()
    return {
        'ram_gb':      mem.rss / 1024**3,
        'ram_percent': psutil.virtual_memory().percent
    }


print('Training LSTM Models')
print('='*80)

for lead in LEAD_TIMES:

    # ----------------------------------------------------------
    # Skip already done leads
    # ----------------------------------------------------------
    if lead in SKIP_LEADS:
        print(f'\n>>> Lead {lead}d — SKIPPED')
        continue

    print(f'\n>>> Lead {lead}d')
    print('-'*80)
    start = time.time()

    # ----------------------------------------------------------
    # Data
    # ----------------------------------------------------------
    train_df_all, train_cols = train_targets
    val_df_all,   _          = val_targets
    test_df_all,  _          = test_targets

    target_col    = f'{TARGET}_lead{lead}'
    train_df_lead = train_df_all
    val_df_lead   = val_df_all
    test_df_lead  = test_df_all

    y_train_vals = train_df_lead[target_col].values.astype(np.int8)
    y_val_vals   = val_df_lead[target_col].values.astype(np.int8)
    y_test_vals  = test_df_lead[target_col].values.astype(np.int8)

    neg = int(np.sum(y_train_vals == 0))
    pos = int(np.sum(y_train_vals == 1))

    print(f"  Class distribution - Train: [{neg}  {pos}]")
    print(f"  MHW rate - Train:      {pos/(neg+pos)*100:.1f}%")
    print(f"  Class distribution - Validation: {np.bincount(y_val_vals)}")
    print(f"  Class distribution - Test: {np.bincount(y_test_vals)}")
    print(f"  MHW rate - Validation: {y_val_vals.mean()*100:.1f}%")
    print(f"  MHW rate - Test:       {y_test_vals.mean()*100:.1f}%")

    # ----------------------------------------------------------
    # Scaler — fit on train only, NaN handled natively
    # ----------------------------------------------------------
    X_train_raw = train_df_lead[FEATURES].values.astype(np.float32)
    scaler      = RobustScaler()
    scaler.fit(X_train_raw)

    # ----------------------------------------------------------
    # Datasets
    # ----------------------------------------------------------
    train_dataset = MaskedLSTMDataset(
        train_df_lead, FEATURES, target_col,
        LSTM_PARAMS['window_size'], LSTM_PARAMS['batch_size'],
        scaler=scaler, mask_value=LSTM_PARAMS['mask_value'],
        shuffle=True
    )
    val_dataset = MaskedLSTMDataset(
        val_df_lead, FEATURES, target_col,
        LSTM_PARAMS['window_size'], LSTM_PARAMS['batch_size'],
        scaler=scaler, mask_value=LSTM_PARAMS['mask_value'],
        shuffle=False
    )
    test_dataset = MaskedLSTMDataset(
        test_df_lead, FEATURES, target_col,
        LSTM_PARAMS['window_size'], LSTM_PARAMS['batch_size'],
        scaler=scaler, mask_value=LSTM_PARAMS['mask_value'],
        shuffle=False
    )

    print(f'  Train: {train_dataset.n_sequences:,} | '
          f'Test: {test_dataset.n_sequences:,}')
    mem = get_memory_usage()
    print(f'  RAM: {mem["ram_gb"]:.2f} GB ({mem["ram_percent"]:.1f}%)')

    # ----------------------------------------------------------
    # Build model
    # ----------------------------------------------------------
    model = build_masked_lstm(
        input_shape  = (LSTM_PARAMS['window_size'], len(FEATURES)),
        hidden_units = LSTM_PARAMS['hidden_units'],
        dropout      = LSTM_PARAMS['dropout'],
        mask_value   = LSTM_PARAMS['mask_value']
    )

    spw   = math.sqrt(neg / pos)
    alpha = spw / (1 + spw)

    model.compile(
        optimizer = keras.optimizers.Adam(
            learning_rate = LSTM_PARAMS['learning_rate'],
            clipnorm      = 1.0
        ),
        loss    = focal_loss(gamma=2.0, alpha=alpha),
        metrics = [keras.metrics.AUC(name='auc', curve='ROC')]
    )
    print(f"  compile done!!  alpha={alpha:.3f}  spw={spw:.2f}")

    # ----------------------------------------------------------
    # Train
    # ----------------------------------------------------------
    history = model.fit(
        train_dataset,
        validation_data = val_dataset,
        epochs          = LSTM_PARAMS['epochs'],
        callbacks = [
            EarlyStopping(
                monitor='val_auc', patience=8,
                restore_best_weights=True,
                mode='max', verbose=1
            ),
            ReduceLROnPlateau(
                monitor='val_auc', factor=0.5,
                patience=5, min_lr=1e-6,
                mode='max', verbose=1
            )
        ],
        verbose = 1
    )
    print("  train done!!")

    # ----------------------------------------------------------
    # Get true labels from dataset
    # ----------------------------------------------------------
    y_val_true = np.array([
        val_dataset.y_data[target_idx]
        for _, target_idx in val_dataset.sequence_indices
    ]).astype(np.int8)

    y_test_true = np.array([
        test_dataset.y_data[target_idx]
        for _, target_idx in test_dataset.sequence_indices
    ]).astype(np.int8)

    # ----------------------------------------------------------
    # Raw predictions
    # ----------------------------------------------------------
    val_prob  = model.predict(val_dataset,  verbose=0).flatten()
    test_prob = model.predict(test_dataset, verbose=0).flatten()

    # ----------------------------------------------------------
    # Calibration
    # ----------------------------------------------------------
    print('\n  Calibrating...')

    calibrator = IsotonicRegression(out_of_bounds='clip')
    calibrator.fit(val_prob, y_val_true)

    cal_val_prob  = calibrator.transform(val_prob)
    cal_test_prob = calibrator.transform(test_prob)

    # Brier score — measures calibration quality
    # Lower = better calibration
    brier_raw = brier_score_loss(y_test_true, test_prob)
    brier_cal = brier_score_loss(y_test_true, cal_test_prob)

    actual_rate = y_val_true.mean()
    print(f'  Val MHW rate:          {actual_rate*100:.1f}%')
    print(f'  Mean raw probability:  {val_prob.mean()*100:.1f}%')
    print(f'  Mean cal probability:  {cal_val_prob.mean()*100:.1f}%')
    print(f'  Calibration works?     '
          f'{"Yes" if abs(cal_val_prob.mean()-actual_rate) < abs(val_prob.mean()-actual_rate) else "⚠️ Check"}')
    print(f'  Brier raw:             {brier_raw:.4f}')
    print(f'  Brier calibrated:      {brier_cal:.4f}')
    print(f'  Brier improvement?     '
          f'{"Yes" if brier_cal < brier_raw else " Warning - Check"}')

    # ----------------------------------------------------------
    # Threshold selection on calibrated val probs
    # ----------------------------------------------------------
    thresholds = np.arange(0.1, 0.9, 0.01)

    # Strategy 1 — MHW F1
    f1_mhw_scores = [
        f1_score(y_val_true,
                 (cal_val_prob > t).astype(int),
                 zero_division=0)
        for t in thresholds
    ]
    best_threshold_mhw = thresholds[np.argmax(f1_mhw_scores)]

    # Strategy 2 — Macro F1
    f1_macro_scores = [
        f1_score(y_val_true,
                 (cal_val_prob > t).astype(int),
                 average='macro', zero_division=0)
        for t in thresholds
    ]
    best_threshold_macro = thresholds[np.argmax(f1_macro_scores)]

    print(f"\n  Threshold (MHW F1):   {best_threshold_mhw:.2f} "
          f"| Val F1: {max(f1_mhw_scores):.3f}")
    print(f"  Threshold (Macro F1): {best_threshold_macro:.2f} "
          f"| Val Macro F1: {max(f1_macro_scores):.3f}")

    # ----------------------------------------------------------
    # Apply thresholds to calibrated test probs
    # ----------------------------------------------------------
    y_pred_mhw   = (cal_test_prob > best_threshold_mhw).astype(int)
    y_pred_macro = (cal_test_prob > best_threshold_macro).astype(int)

    cm_mhw   = confusion_matrix(y_test_true, y_pred_mhw)
    cm_macro = confusion_matrix(y_test_true, y_pred_macro)
    tn_m, fp_m, fn_m, tp_m = cm_mhw.ravel()
    tn_b, fp_b, fn_b, tp_b = cm_macro.ravel()

    # ----------------------------------------------------------
    # Metrics
    # ----------------------------------------------------------
    result_mhw = compute_metrics(
        y_test_true, y_pred_mhw, cal_test_prob,
        best_threshold_mhw, 'mhw_f1',
        'LSTM', CONFIG_NAME, TARGET, lead
    )
    result_macro = compute_metrics(
        y_test_true, y_pred_macro, cal_test_prob,
        best_threshold_macro, 'macro_f1',
        'LSTM', CONFIG_NAME, TARGET, lead
    )

    best_epoch   = np.argmax(history.history['val_auc']) + 1
    best_val_auc = max(history.history['val_auc'])

    # Add extra fields
    for r in [result_mhw, result_macro]:
        r['best_epoch']        = best_epoch
        r['best_val_auc']      = best_val_auc
        r['calibrated']        = True
        r['brier_raw']         = float(brier_raw)
        r['brier_calibrated']  = float(brier_cal)

    # ----------------------------------------------------------
    # Print —
    # ----------------------------------------------------------
    print('\n=== MHW F1 THRESHOLD (Detection Focused) ===')
    print(classification_report(
        y_test_true, y_pred_mhw,
        target_names=['No MHW (0)', 'MHW (1)'],
        digits=3
    ))
    print(f'    TN={tn_m:,} | FP={fp_m:,}')
    print(f'    FN={fn_m:,} | TP={tp_m:,}')
    print(f'    TPR={result_mhw["tpr"]:.3f} | '
          f'TNR={result_mhw["tnr"]:.3f}')
    print(f'    FPR={result_mhw["fpr"]:.3f} | '
          f'CSI={result_mhw["csi"]:.3f}')

    print('\n=== MACRO F1 THRESHOLD (Balanced) ===')
    print(classification_report(
        y_test_true, y_pred_macro,
        target_names=['No MHW (0)', 'MHW (1)'],
        digits=3
    ))
    print(f'    TN={tn_b:,} | FP={fp_b:,}')
    print(f'    FN={fn_b:,} | TP={tp_b:,}')
    print(f'    TPR={result_macro["tpr"]:.3f} | '
          f'TNR={result_macro["tnr"]:.3f}')
    print(f'    FPR={result_macro["fpr"]:.3f} | '
          f'CSI={result_macro["csi"]:.3f}')

    elapsed = time.time() - start
    print(f'\nMHW F1:   F1={result_mhw["f1"]:.3f} '
          f'AUC={result_mhw["auc"]:.3f} '
          f'FPR={result_mhw["fpr"]:.3f} '
          f'TPR={result_mhw["tpr"]:.3f}')
    print(f'Macro F1: F1={result_macro["macro_f1"]:.3f} '
          f'AUC={result_macro["auc"]:.3f} '
          f'FPR={result_macro["fpr"]:.3f} '
          f'TPR={result_macro["tpr"]:.3f}')
    print(f'Brier:    raw={brier_raw:.4f} → '
          f'calibrated={brier_cal:.4f} '
          f'{"good" if brier_cal < brier_raw else "warning"}')
    print(f'Best epoch: {best_epoch} | '
          f'Best val AUC: {best_val_auc:.3f} | '
          f'Time: {elapsed:.1f}s')

    # ----------------------------------------------------------
    # SAVE PER LEAD
    # ----------------------------------------------------------
    print(f'\n  Saving lead {lead}d files...')

    # Model
    model.save(
        OUTPUT_DIR / f'lstm_{CONFIG_NAME}_lead{lead}_model.keras'
    )
    # Scaler
    with open(OUTPUT_DIR / f'lstm_{CONFIG_NAME}_lead{lead}_scaler.pkl', 'wb') as f:
        pickle.dump(scaler, f)
    # Calibrator
    with open(OUTPUT_DIR / f'lstm_{CONFIG_NAME}_lead{lead}_calibrator.pkl', 'wb') as f:
        pickle.dump(calibrator, f)
    # History
    with open(OUTPUT_DIR / f'lstm_{CONFIG_NAME}_lead{lead}_history.pkl', 'wb') as f:
        pickle.dump(history.history, f)
    # Results for this lead
    pd.DataFrame([result_mhw, result_macro]).to_csv(
        OUTPUT_DIR / f'lstm_{CONFIG_NAME}_lead{lead}_results.csv',
        index=False
    )

    print(f'    lstm_{CONFIG_NAME}_lead{lead}_model.keras      ')
    print(f'    lstm_{CONFIG_NAME}_lead{lead}_scaler.pkl    ')
    print(f'    lstm_{CONFIG_NAME}_lead{lead}_calibrator.pkl')
    print(f'    lstm_{CONFIG_NAME}_lead{lead}_history.pkl   ')
    print(f'    lstm_{CONFIG_NAME}_lead{lead}_results.csv   ')
    print(f'\n  Lead {lead}d complete')

    # ----------------------------------------------------------
    # Cleanup
    # ----------------------------------------------------------
    del train_dataset, val_dataset, test_dataset, model
    keras.backend.clear_session()
    gc.collect()
    if tf.config.list_physical_devices('GPU'):
        tf.config.experimental.reset_memory_stats('GPU:0')

    mem_end = get_memory_usage()
    print(f'  RAM: {mem_end["ram_gb"]:.2f} GB '
          f'({mem_end["ram_percent"]:.1f}%)')
    print('-'*80)

print('\nAll leads done ')
print('='*80)

Training LSTM Models

>>> Lead 3d
--------------------------------------------------------------------------------
  Class distribution - Train: [8584132  977305]
  MHW rate - Train:      10.2%
  Class distribution - Validation: [828738 208290]
  Class distribution - Test: [476276 560752]
  MHW rate - Validation: 20.1%
  MHW rate - Test:       54.1%
  Converting DataFrame to NumPy...
  Building sequence indices...
  Dataset ready: 9,546,872 sequences
    X array:  401.2 MB
    y array:  36.5 MB
    Indices:  1092.6 MB
    Batches:  4,662
  Converting DataFrame to NumPy...
  Building sequence indices...
  Dataset ready: 1,022,463 sequences
    X array:  43.5 MB
    y array:  4.0 MB
    Indices:  117.0 MB
    Batches:  500
  Converting DataFrame to NumPy...
  Building sequence indices...
  Dataset ready: 1,022,463 sequences
    X array:  43.5 MB
    y array:  4.0 MB
    Indices:  117.0 MB
    Batches:  500
  Train: 9,546,872 | Test: 1,022,463
  RAM: 9.52 GB (7.2%)
  compile done!!  alpha

In [15]:

models  = {}
scalers = {}

print('Loading models and scalers...')

for lead in LEAD_TIMES:
    # Load model
    model_path = OUTPUT_DIR / f'lstm_{CONFIG_NAME}_lead{lead}_model.keras'
    if model_path.exists():
        models[lead] = keras.models.load_model(
            model_path,
            custom_objects={'loss_fn': focal_loss()}
        )
        print(f'  Lead {lead}d model')
    else:
        print(f'  Lead {lead}d model missing')

    # Load scaler
    scaler_path = OUTPUT_DIR / f'lstm_{CONFIG_NAME}_lead{lead}_scaler.pkl'
    if scaler_path.exists():
        with open(scaler_path, 'rb') as f:
            scalers[lead] = pickle.load(f)
        print(f'  Lead {lead}d scaler')
    else:
        print(f'  Lead {lead}d scaler missing')

print(f'\nLoaded {len(models)} models, {len(scalers)} scalers')
print('Now run SHAP cell')

Loading models and scalers...
  Lead 3d model
  Lead 3d scaler
  Lead 5d model
  Lead 5d scaler
  Lead 7d model
  Lead 7d scaler
  Lead 9d model
  Lead 9d scaler
  Lead 12d model
  Lead 12d scaler
  Lead 14d model
  Lead 14d scaler

Loaded 6 models, 6 scalers
Now run SHAP cell


In [16]:
# =====================================================================
# LSTM SHAP — random sequences across locations
# =====================================================================

N_BG   = 1000   # background sequences (training period)
N_TEST = 1000    # test sequences explained
N_LOCS = 50     # random locations to draw from
SEED   = 42


def draw_random_sequences(ds, n, rng):
    """Pick n random sequences from a MaskedLSTMDataset and return them
    scaled and masked exactly as __getitem__ does."""
    n   = min(n, ds.n_sequences)
    sel = rng.choice(ds.n_sequences, size=n, replace=False)
    ds.sequence_indices = [ds.sequence_indices[i] for i in sel]
    ds.n_sequences      = n
    ds.batch_size       = n          # one batch = the whole sample
    X, _ = ds[0]                     # same scaling + mask_value as training
    locs = ds.locations[[t for _, t in ds.sequence_indices]]
    return X, locs


print('SHAP Feature Importance Analysis (random sequences)')
print(f'Background: {N_BG} | Test: {N_TEST} | Locations: {N_LOCS} | Seed: {SEED}')
print('='*80)

shap_data       = {}
importance_data = []

for lead in LEAD_TIMES:
    print(f'\n>>> Lead {lead}d')
    print('-'*80)

    train_df_all, train_cols = train_targets
    test_df_all,  _          = test_targets
    target_col    = f'{TARGET}_lead{lead}'
    train_df_lead = train_df_all
    test_df_lead  = test_df_all
    scaler        = scalers[lead]
    model         = models[lead]

    rng = np.random.default_rng(SEED)

    # Background — random sequences from N_LOCS random training locations
    all_locs  = train_df_lead['location'].unique()
    bg_locs   = rng.choice(all_locs, size=min(N_LOCS, len(all_locs)),
                           replace=False)
    bg_sample = train_df_lead[train_df_lead['location'].isin(bg_locs)]

    bg_dataset = MaskedLSTMDataset(
        bg_sample, FEATURES, target_col,
        LSTM_PARAMS['window_size'], 500,
        scaler=scaler,
        mask_value=LSTM_PARAMS['mask_value'],
        shuffle=False
    )
    X_background, bg_seq_locs = draw_random_sequences(bg_dataset, N_BG, rng)
    print(f'  X_background: {X_background.shape} | '
          f'locations used: {len(np.unique(bg_seq_locs))}')

    # Test sample — random sequences from N_LOCS random test locations
    all_locs_test = test_df_lead['location'].unique()
    test_locs     = rng.choice(all_locs_test,
                               size=min(N_LOCS, len(all_locs_test)),
                               replace=False)
    test_sample   = test_df_lead[test_df_lead['location'].isin(test_locs)]

    test_ds = MaskedLSTMDataset(
        test_sample, FEATURES, target_col,
        LSTM_PARAMS['window_size'], 500,
        scaler=scaler,
        mask_value=LSTM_PARAMS['mask_value'],
        shuffle=False
    )
    X_sample_small, test_seq_locs = draw_random_sequences(test_ds, N_TEST, rng)
    print(f'  X_sample: {X_sample_small.shape} | '
          f'locations used: {len(np.unique(test_seq_locs))}')

    # SHAP computation
    print('  Computing SHAP...')
    shap_values = None
    method_used = None

    # Try 1 — GradientExplainer
    try:
        explainer   = shap.GradientExplainer(model, X_background)
        shap_values = explainer.shap_values(X_sample_small)
        method_used = 'GradientExplainer'
    except Exception as e:
        print(f'  GradientExplainer failed: {e}')

    # Try 2 — DeepExplainer
    if shap_values is None:
        try:
            explainer   = shap.DeepExplainer(model, X_background)
            shap_values = explainer.shap_values(X_sample_small)
            method_used = 'DeepExplainer'
        except Exception as e:
            print(f'  DeepExplainer failed: {e}')

    # Try 3 — KernelExplainer (slow fallback)
    if shap_values is None:
        print('  Method: KernelExplainer (slower)...')
        n_t = X_background.shape[1]
        n_f = X_background.shape[2]
        n_k = min(100, len(X_background))

        X_bg_flat   = X_background[:n_k].reshape(n_k, n_t*n_f)
        X_samp_flat = X_sample_small.reshape(len(X_sample_small), n_t*n_f)

        def model_predict(X_flat):
            X_3d = X_flat.reshape(-1, n_t, n_f)
            return model.predict(X_3d, verbose=0).flatten()

        explainer = shap.KernelExplainer(model_predict, X_bg_flat,
                                         link='logit')
        shap_values_flat = explainer.shap_values(X_samp_flat, nsamples=100,
                                                 silent=True)
        shap_values = np.asarray(shap_values_flat).reshape(
            len(X_sample_small), n_t, n_f)
        method_used = 'KernelExplainer'

    if shap_values is None:
        print('  All SHAP methods failed — skipping lead')
        continue
    print(f'  Method: {method_used}')

    # Get positive class
    if isinstance(shap_values, list):
        shap_values_positive = shap_values[1] if len(shap_values) > 1 else shap_values[0]
    else:
        shap_values_positive = shap_values
    shap_values_positive = np.asarray(shap_values_positive)
    # GradientExplainer can return (n, t, f, 1) for a single output
    if shap_values_positive.ndim == 4 and shap_values_positive.shape[-1] == 1:
        shap_values_positive = shap_values_positive[..., 0]
    print(f'  SHAP shape: {shap_values_positive.shape}')

    # Aggregate: mean |SHAP| over samples and timesteps
    mean_shap = np.abs(shap_values_positive).mean(axis=(0, 1))

    feature_importance = {col: float(v) for col, v in zip(FEATURES, mean_shap)}
    total = sum(feature_importance.values())
    feature_importance = {k: (v/total)*100 for k, v in feature_importance.items()}

    sorted_features = sorted(feature_importance.items(),
                             key=lambda x: x[1], reverse=True)

    print('  Features by SHAP:')
    for rank, (feat, imp) in enumerate(sorted_features, 1):
        print(f'    {rank}. {feat:20s} {imp:5.1f}%')
        importance_data.append({
            'config':          CONFIG_NAME,
            'target':          TARGET,
            'lead_time':       lead,
            'feature':         feat,
            'shap_importance': imp,
            'rank':            rank,
            'n_background':    len(X_background),
            'n_test':          len(X_sample_small),
            'n_bg_locations':  int(len(np.unique(bg_seq_locs))),
            'n_test_locations': int(len(np.unique(test_seq_locs))),
            'shap_method':     method_used
        })

    shap_data[lead] = {
        'shap_values':     shap_values_positive,
        'feature_names':   FEATURES,
        'base_features':   list(feature_importance.keys()),
        'base_importance': list(feature_importance.values())
    }

    del bg_dataset, test_ds, X_background, X_sample_small
    gc.collect()
    print('-'*80)

# Save (same file names as before — back up the old files first)
importance_df = pd.DataFrame(importance_data)
importance_df.to_csv(
    OUTPUT_DIR / f'lstm_{CONFIG_NAME}_shap_importance.csv',
    index=False
)

with open(OUTPUT_DIR / f'lstm_{CONFIG_NAME}_shap_values.pkl', 'wb') as f:
    pickle.dump(shap_data, f)

print('\nSHAP saved')
print('='*80)

SHAP Feature Importance Analysis (random sequences)
Background: 1000 | Test: 1000 | Locations: 50 | Seed: 42

>>> Lead 3d
--------------------------------------------------------------------------------
  Converting DataFrame to NumPy...
  Building sequence indices...
  Dataset ready: 491,600 sequences
    X array:  20.7 MB
    y array:  1.9 MB
    Indices:  56.3 MB
    Batches:  984
  X_background: (1000, 15, 11) | locations used: 50
  Converting DataFrame to NumPy...
  Building sequence indices...
  Dataset ready: 52,650 sequences
    X array:  2.2 MB
    y array:  0.2 MB
    Indices:  6.0 MB
    Batches:  106
  X_sample: (1000, 15, 11) | locations used: 50
  Computing SHAP...
  Method: GradientExplainer
  SHAP shape: (1000, 15, 11)
  Features by SHAP:
    1. temp_0_50m            30.5%
    2. sst                   15.3%
    3. temp_50_100m          13.6%
    4. temp_100_150m          9.4%
    5. ssr_flux               9.0%
    6. t2m                    6.7%
    7. latent_flux       

In [ ]:
# ==============================================================
# FINAL CELL — Combine results only
# ==============================================================

print('Combining results...')

all_results = []

for lead in LEAD_TIMES:
    path = OUTPUT_DIR / f'lstm_{CONFIG_NAME}_lead{lead}_results.csv'
    if path.exists():
        all_results.append(pd.read_csv(path))
        print(f'  Lead {lead}d')
    else:
        print(f'  Lead {lead}d missing — check training')

all_results_df = pd.concat(all_results, ignore_index=True)
all_results_df.to_csv(
    OUTPUT_DIR / f'lstm_{CONFIG_NAME}_results.csv',
    index=False
)

print(f'\nCombined results saved')
print(f'   Rows: {len(all_results_df)}')
print(f'   Leads: {all_results_df["lead_time"].unique().tolist()}')

Combining results...
  Lead 3d
  Lead 5d
  Lead 7d
  Lead 9d
  Lead 12d
  Lead 14d

Combined results saved
   Rows: 12
   Leads: [3, 5, 7, 9, 12, 14]


In [ ]:
# Save Results and Metadata
print('Saving results...')

# # Results
# results_df   = pd.DataFrame(results)
# results_path = OUTPUT_DIR / f'lstm_{CONFIG_NAME}_results.csv'
# results_df.to_csv(results_path, index=False)
# print(f'Results: {results_path.name}')

# # Scalers (CRITICAL!)
# scalers_path = OUTPUT_DIR / f'lstm_{CONFIG_NAME}_scalers.pkl'
# with open(scalers_path, 'wb') as f:
#     pickle.dump(scalers, f)
# print(f'Scalers: {scalers_path.name}')

# # Histories
# histories_path = OUTPUT_DIR / f'lstm_{CONFIG_NAME}_histories.pkl'
# with open(histories_path, 'wb') as f:
#     pickle.dump(histories, f)
# print(f'Histories: {histories_path.name}')

# Separate mhw_f1 and macro_f1 rows for averages
mhw_df   = all_results_df[all_results_df['threshold_strategy'] == 'mhw_f1']
macro_df = all_results_df[all_results_df['threshold_strategy'] == 'macro_f1']

# Metadata — updated to match fixes
metadata = {
    'date_created':       pd.Timestamp.now().isoformat(),
    'model_type':         'LSTM',
    'config':             CONFIG_NAME,
    'features':           FEATURES,
    'target':             TARGET,
    'lead_times':         LEAD_TIMES,
    'params':             LSTM_PARAMS,
    'scaling':            'RobustScaler — NaN handled natively',
    'nan_handling':       'Sentinel value -10.0 after scaling',
    'mask_value':         LSTM_PARAMS['mask_value'],
    'feature_importance': 'SHAP GradientExplainer',
    'threshold_strategies': ['mhw_f1', 'macro_f1'],
    # MHW F1 threshold averages
    'avg_auc':            float(mhw_df['auc'].mean()),
    'avg_f1_mhw':         float(mhw_df['f1'].mean()),
    'avg_tpr_mhw':        float(mhw_df['tpr'].mean()),
    'avg_fpr_mhw':        float(mhw_df['fpr'].mean()),
    # Macro F1 threshold averages
    'avg_macro_f1':       float(macro_df['macro_f1'].mean()),
    'avg_tpr_macro':      float(macro_df['tpr'].mean()),
    'avg_fpr_macro':      float(macro_df['fpr'].mean()),
}

metadata_path = OUTPUT_DIR / f'lstm_{CONFIG_NAME}_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'Metadata: {metadata_path.name}')

print('\n' + '='*80)
print('SUMMARY')
print('='*80)
print(f'Configuration: {CONFIG_NAME}')
print(f'Models trained: {len(models)}')

print(f'\n  MHW F1 threshold strategy:')
print(f'    Average AUC:  {mhw_df["auc"].mean():.3f}')
print(f'    Average F1:   {mhw_df["f1"].mean():.3f}')
print(f'    Average TPR:  {mhw_df["tpr"].mean():.3f}')
print(f'    Average FPR:  {mhw_df["fpr"].mean():.3f}')

print(f'\n  Macro F1 threshold strategy:')
print(f'    Average AUC:       {macro_df["auc"].mean():.3f}')
print(f'    Average Macro F1:  {macro_df["macro_f1"].mean():.3f}')
print(f'    Average TPR:       {macro_df["tpr"].mean():.3f}')
print(f'    Average FPR:       {macro_df["fpr"].mean():.3f}')

print('\nFiles saved:')
print(f'  - {len(models)} model files (.keras)')
print(f'  - 1 scalers file (.pkl) ')
print(f'  - 1 results file (.csv)')
print(f'  - 1 histories file (.pkl)')
print(f'  - 1 metadata file (.json)')
print(f'  - 1 SHAP importance (.csv)')
print(f'  - 1 SHAP values (.pkl)')

print('\n  Scaling:      RobustScaler')
print('  NaN handling: Sentinel -10.0')
print('  Threshold:    Optimised on val set ')
print('='*80)

Saving results...
Metadata: lstm_subsurface_layer_0-50m_layer_metadata.json

SUMMARY
Configuration: subsurface_layer_0-50m_layer
Models trained: 6

  MHW F1 threshold strategy:
    Average AUC:  0.762
    Average F1:   0.737
    Average TPR:  0.841
    Average FPR:  0.524

  Macro F1 threshold strategy:
    Average AUC:       0.762
    Average Macro F1:  0.679
    Average TPR:       0.740
    Average FPR:       0.384

Files saved:
  - 6 model files (.keras)
  - 1 scalers file (.pkl) 
  - 1 results file (.csv)
  - 1 histories file (.pkl)
  - 1 metadata file (.json)
  - 1 SHAP importance (.csv)
  - 1 SHAP values (.pkl)

  Scaling:      RobustScaler
  NaN handling: Sentinel -10.0
  Threshold:    Optimised on val set 
